# Train on Colab → Export → Download → Run Locally

This notebook:
1. Sets up the `real_time_object_tracking` project on a Colab GPU runtime.
2. Downloads a curated COCO subset (fast `demo` mode, or the bigger `full_subset` mode).
3. Fine-tunes the YOLOv8n detector.
4. Evaluates it (mAP / precision / recall / FPS).
5. Exports the checkpoint (`.pt`, and optionally `.onnx`) and zips it up.
6. Downloads the zip to your computer (or saves it to Google Drive).

**Before running:** in Colab, go to `Runtime → Change runtime type → GPU` (T4 is fine).

At the end you'll have a small zip containing `best.pt` (and optionally `best.onnx`) --
copy that into `artifacts/exported_models/` in your **local** copy of this project and
you're ready to run inference / the Streamlit app / the API locally, without needing
a GPU or retraining anything.

## 0. Confirm you have a GPU

In [ ]:
!nvidia-smi

## 1. Get the project onto the Colab machine

Pick **ONE** of the two options below.

**Option A -- Upload the zip you already have** (the one from this chat / your repo export).
Run the cell, then use the file picker to select `real_time_object_tracking.zip`.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select real_time_object_tracking.zip in the dialog
zip_name = next(iter(uploaded.keys()))
print("Uploaded:", zip_name)

import zipfile
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(".")
print("Extracted.")


**Option B -- Clone from GitHub** (skip this cell if you used Option A above).
Only works if you've pushed this project to a GitHub repo.

In [ ]:
# Uncomment and edit if you're cloning from GitHub instead of uploading a zip:
# !git clone https://github.com/<your-username>/real_time_object_tracking.git


In [ ]:
%cd real_time_object_tracking
!ls

## 2. Install dependencies

Colab already ships with `torch`, but `pip install -e .` will make sure every
dependency in `pyproject.toml` (ultralytics, opencv, pydantic, etc.) is present
and importable as the `object_tracking_app` package.

In [ ]:
!pip install -q -e .


In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 3. (Recommended) Mount Google Drive for persistent storage

Colab's local disk is wiped when the runtime disconnects. Mounting Drive lets you
cache the downloaded COCO subset and checkpoints so you don't have to re-download
everything if your session restarts. Skip this cell if you don't want to use Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point the dataset raw/processed dirs at Drive so downloads persist across sessions.
import os
DRIVE_DATA_DIR = "/content/drive/MyDrive/real_time_object_tracking_data"
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

# Symlink data/ to the Drive location (comment this out if you'd rather use local disk).
!rm -rf data
!ln -s "$DRIVE_DATA_DIR" data
!mkdir -p data/raw data/processed data/external data/splits data/samples
print("data/ now points to:", DRIVE_DATA_DIR)


## 4. Download the COCO subset

- `demo` mode: ~200 images, trains in minutes -- good for a first end-to-end smoke test.
- `full_subset` mode: ~8000 images, much better accuracy, longer download + training time.

Start with `demo` to confirm everything works, then switch to `full_subset` for a real run.

In [ ]:
DATASET_MODE = "demo"  # change to "full_subset" for a bigger, better-quality run

!python main.py download-data --mode {DATASET_MODE}


## 5. Train

Adjust `EPOCHS` / `BATCH_SIZE` as needed. `demo` mode with `yolov8n` trains in well
under 30 minutes on a T4. For `full_subset`, expect roughly 1-3 hours depending on
epochs -- consider using Drive (step 3) so a disconnect doesn't lose your dataset.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 16

!python main.py train --mode {DATASET_MODE} --epochs {EPOCHS} --batch-size {BATCH_SIZE}


## 6. Evaluate the trained checkpoint

In [ ]:
!python main.py evaluate

import json
with open("artifacts/metrics/evaluation_report.json") as f:
    print(json.dumps(json.load(f), indent=2))


## 7. (Optional) Export to ONNX

Useful if you want to run inference later without `ultralytics`/`torch` installed
(e.g. with `onnxruntime` on a lighter machine).

In [ ]:
!python main.py export --format onnx


## 8. Zip up everything you need locally

This grabs:
- `artifacts/exported_models/best.pt` (and `best.onnx` if you ran the export cell)
- `artifacts/metrics/evaluation_report.json` (your benchmark numbers)
- `configs/` (so your local copy's settings match what you trained with, in case
  you changed `configs/train.yaml` / `configs/data.yaml` here)

In [ ]:
import shutil
from pathlib import Path

bundle_dir = Path("trained_model_bundle")
bundle_dir.mkdir(exist_ok=True)

# Checkpoints
exported_dir = Path("artifacts/exported_models")
for pattern in ("best.pt", "best.onnx", "*.onnx"):
    for f in exported_dir.glob(pattern):
        shutil.copy2(f, bundle_dir / f.name)

# Metrics
metrics_src = Path("artifacts/metrics/evaluation_report.json")
if metrics_src.exists():
    shutil.copy2(metrics_src, bundle_dir / "evaluation_report.json")

# Configs (so local settings match)
shutil.copytree("configs", bundle_dir / "configs", dirs_exist_ok=True)

zip_path = shutil.make_archive("trained_model_bundle", "zip", root_dir=".", base_dir="trained_model_bundle")
print("Created:", zip_path)
!ls -la trained_model_bundle


## 9. Download the bundle to your computer

Triggers a browser download of `trained_model_bundle.zip`.

In [ ]:
from google.colab import files
files.download("trained_model_bundle.zip")


### Alternative: save the bundle to Google Drive instead of downloading

Handy if the direct-download dialog gets blocked by your browser, or you just want
a backup.

In [ ]:
# import shutil
# shutil.copy2("trained_model_bundle.zip", "/content/drive/MyDrive/trained_model_bundle.zip")
# print("Saved to Google Drive: MyDrive/trained_model_bundle.zip")


## 10. Use it locally

On your own machine, inside your local copy of `real_time_object_tracking/`:

```bash
# 1. Unzip the bundle you just downloaded
unzip trained_model_bundle.zip -d /tmp/trained_model_bundle

# 2. Copy the checkpoint into place
cp /tmp/trained_model_bundle/best.pt artifacts/exported_models/best.pt

# (Optional) if you exported ONNX too:
cp /tmp/trained_model_bundle/best.onnx artifacts/exported_models/best.onnx

# 3. (Optional) if you changed configs/train.yaml or configs/data.yaml on Colab,
#    copy those over too so class lists / thresholds match what the model was trained on:
cp /tmp/trained_model_bundle/configs/*.yaml configs/

# 4. Sync your local environment (only needed once)
uv sync

# 5. Run inference with your fine-tuned model
uv run python main.py infer-image --source data/samples/example.jpg --track

# 6. Or launch the full deployment UI
uv run python main.py run-app
```

`Detector._resolve_weights()` automatically picks up
`artifacts/exported_models/best.pt` if it exists (falling back to the pretrained
`yolov8n.pt` otherwise), so no code changes are needed -- just drop the file in
place.